# Module 1: Spark Basics

## What is Apache Spark?

Apache Spark is a **distributed computing engine**. It does one thing exceptionally well: take a computation that would be too slow or too large for a single machine and spread it across a cluster of machines, running it in parallel.

**Why should you care?** Because in the real world, data grows faster than single machines get faster. A retailer might have 10TB of transaction logs. A social platform generates petabytes daily. Spark lets you process this data in minutes instead of days.

### Key ideas you need to internalize:

- **Driver + Executors**: Your program (the driver) tells worker processes (executors) what to do. In local mode, they all live in one JVM on your laptop.
- **In-memory processing**: Spark keeps data in RAM between operations, unlike Hadoop MapReduce which writes to disk after every step. This makes Spark 10-100x faster for iterative workloads.
- **Lazy evaluation**: When you write `df.filter(...)`, Spark does NOT run it immediately. It builds a plan (a DAG). Only when you call an action like `.show()` does it optimize the entire plan and execute it.
- **Spark is a compute engine, not storage**: Spark reads data from storage (CSV, S3, HDFS, databases), processes it in memory, and writes results back. It doesn't store your data permanently.

---
## Concept 1: Creating a SparkSession

The `SparkSession` is your **entry point** to everything in Spark. Think of it like opening a database connection — nothing happens until you have one.

In production, this connects to a real cluster with hundreds of machines. In local mode, it creates a mini-cluster inside your laptop.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Module 01 - Spark Basics") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Spark UI: http://localhost:4040")

Spark version: 4.1.1
Spark UI: http://localhost:4040


**What each line does:**
- `.appName("...")` — names this job (you'll see it in the Spark UI)
- `.master("local[*]")` — run locally using ALL available CPU cores. `local[2]` would use only 2 cores.
- `.getOrCreate()` — creates a new session, or reuses one if it already exists

#### Try It: Create Your Own SparkSession

Stop the current session with `spark.stop()`, then create a new one with:
- App name: `"My First Spark App"`
- Master: `"local[2]"` (only 2 cores)
- Print the Spark version

In [3]:
# Your code here
spark.stop()

spark = SparkSession.builder \
    .appName("My First Spark App") \
    .master("local[2]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")


Spark version: 4.1.1


In [ ]:
# --- Solution ---
spark.stop()

spark = SparkSession.builder \
    .appName("My First Spark App") \
    .master("local[2]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

---
## Concept 2: The SparkContext

Under the SparkSession lives the `SparkContext` — the lower-level connection to the cluster. You'll use it for RDD operations (Module 2). For now, it tells you about your environment.

In a production cluster, `defaultParallelism` would match your total executor cores (e.g., 200 cores across 50 machines). Locally, it matches your CPU cores.

In [4]:
sc = spark.sparkContext

print(f"App name: {sc.appName}")
print(f"Master: {sc.master}")
print(f"Default parallelism: {sc.defaultParallelism}")

App name: My First Spark App
Master: local[2]
Default parallelism: 2


#### Try It: Explore Your SparkContext

1. Print `sc.defaultParallelism` — how many cores is Spark using? (Hint: you set `local[2]` above)
2. Print `sc.version` — does it match `spark.version`?
3. Print `sc.uiWebUrl` — this is the URL for the Spark UI

In [5]:
# Your code here



In [6]:
# --- Solution ---
print(f"Parallelism: {sc.defaultParallelism}")
print(f"SC version: {sc.version}")
print(f"Spark version: {spark.version}")
print(f"UI URL: {sc.uiWebUrl}")

Parallelism: 2
SC version: 4.1.1
Spark version: 4.1.1
UI URL: http://b6414d5da7e5:4040


---
## Concept 3: Creating DataFrames from Python Data

A DataFrame is Spark's **core data structure** — like a table in a database or a pandas DataFrame, but distributed across the cluster.

**Why not just use pandas?** Pandas loads everything into one machine's RAM. If your data is 100GB and your machine has 16GB RAM, pandas crashes. A Spark DataFrame splits that 100GB across the cluster.

For learning, we create small DataFrames from Python lists. In production, you'd read from files, databases, or streaming sources.

In [7]:
data = [
    ("Alice", 30, "San Francisco"),
    ("Bob", 25, "New York"),
    ("Carol", 35, "Chicago"),
]

df = spark.createDataFrame(data, ["name", "age", "city"])

# .show() prints the table (this is an ACTION — triggers execution)
df.show()

# .printSchema() shows column names and types
df.printSchema()

+-----+---+-------------+
| name|age|         city|
+-----+---+-------------+
|Alice| 30|San Francisco|
|  Bob| 25|     New York|
|Carol| 35|      Chicago|
+-----+---+-------------+

root
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- city: string (nullable = true)



Notice: Spark inferred types automatically — `name` is `string`, `age` is `long` (not int!), `city` is `string`.

#### Try It: Build Your Own DataFrame

Create a DataFrame with **5 rows** and columns: `name` (string), `score` (int), `subject` (string).

Display it with `.show()` and check the schema with `.printSchema()`.

In [8]:
# Your code here
data = [
    ("Mike", 10, "Big Data"),
    ("Ral", 4, "AI"),
    ("Kim", 3, "ML"),
    ("Ted", 2, "CV"),
    ("Kas", 1, "DM"),
]

df = spark.createDataFrame(data, ["name", "score", "subject"])

df.show()

df.printSchema()

+----+-----+--------+
|name|score| subject|
+----+-----+--------+
|Mike|   10|Big Data|
| Ral|    4|      AI|
| Kim|    3|      ML|
| Ted|    2|      CV|
| Kas|    1|      DM|
+----+-----+--------+

root
 |-- name: string (nullable = true)
 |-- score: long (nullable = true)
 |-- subject: string (nullable = true)



In [9]:
# --- Solution ---
students = [
    ("Alice", 92, "Math"),
    ("Bob", 85, "Science"),
    ("Carol", 78, "Math"),
    ("Dave", 91, "English"),
    ("Eve", 88, "Science"),
]
student_df = spark.createDataFrame(students, ["name", "score", "subject"])
student_df.show()
student_df.printSchema()

+-----+-----+-------+
| name|score|subject|
+-----+-----+-------+
|Alice|   92|   Math|
|  Bob|   85|Science|
|Carol|   78|   Math|
| Dave|   91|English|
|  Eve|   88|Science|
+-----+-----+-------+

root
 |-- name: string (nullable = true)
 |-- score: long (nullable = true)
 |-- subject: string (nullable = true)



---
## Concept 4: Reading CSV Files

In the real world, you don't type data by hand. You read it from files. Spark can read CSV, JSON, Parquet, ORC, Avro, JDBC databases, and more.

CSV is the simplest format — human-readable rows of comma-separated values. But it's also the **worst** format for Big Data (no schema, no compression, slow to read). You'll learn better formats in Module 6.

Two important options when reading CSV:
- `header=True` — the first row contains column names
- `inferSchema=True` — Spark reads the file to guess column types (slower but convenient)

In [10]:
employees = spark.read.csv("../data/employees.csv", header=True, inferSchema=True)

print(f"Rows: {employees.count()}")
print(f"Columns: {employees.columns}")
employees.show(10)

Rows: 30
Columns: ['employee_id', 'name', 'department_id', 'salary', 'hire_date', 'city']
+-----------+-------------+-------------+------+----------+-------------+
|employee_id|         name|department_id|salary| hire_date|         city|
+-----------+-------------+-------------+------+----------+-------------+
|          1|   Alice Chen|          101| 92000|2019-03-15|San Francisco|
|          2| Bob Martinez|          102| 78000|2020-07-01|     New York|
|          3|Carol Johnson|          101|105000|2017-11-20|San Francisco|
|          4|    David Kim|          103| 67000|2021-01-10|      Chicago|
|          5|Eve Rodriguez|          104| 72000|2020-09-05|     New York|
|          6| Frank Wilson|          105| 95000|2018-04-22|San Francisco|
|          7|    Grace Lee|          101|110000|2016-08-30|San Francisco|
|          8|  Henry Brown|          106| 62000|2022-03-14|      Chicago|
|          9|  Irene Davis|          102| 83000|2019-06-18|     New York|
|         10|Jack Thom

In [11]:
employees.printSchema()

root
 |-- employee_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department_id: integer (nullable = true)
 |-- salary: integer (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- city: string (nullable = true)



#### Try It: Read Another CSV

1. Read `../data/departments.csv` into a DataFrame called `departments`
2. Print how many rows it has
3. Display ALL rows with `.show()`
4. Print the schema

In [15]:
# Your code here
departments_df = spark.read.csv("../data/departments.csv", header=True, inferSchema=True)

print(f"Rows: {departments_df.count()}")
print(f"Comumns: {departments_df.columns}")

departments_df.show()


Rows: 6
Comumns: ['department_id', 'department_name', 'budget', 'location']
+-------------+---------------+-------+-------------+
|department_id|department_name| budget|     location|
+-------------+---------------+-------+-------------+
|          101|    Engineering|2500000|San Francisco|
|          102|      Marketing|1200000|     New York|
|          103|          Sales|1800000|      Chicago|
|          104|Human Resources| 800000|     New York|
|          105|        Finance|1500000|San Francisco|
|          106|     Operations| 950000|      Chicago|
+-------------+---------------+-------+-------------+



In [16]:
# --- Solution ---
departments = spark.read.csv("../data/departments.csv", header=True, inferSchema=True)
print(f"Rows: {departments.count()}")
departments.show()
departments.printSchema()

Rows: 6
+-------------+---------------+-------+-------------+
|department_id|department_name| budget|     location|
+-------------+---------------+-------+-------------+
|          101|    Engineering|2500000|San Francisco|
|          102|      Marketing|1200000|     New York|
|          103|          Sales|1800000|      Chicago|
|          104|Human Resources| 800000|     New York|
|          105|        Finance|1500000|San Francisco|
|          106|     Operations| 950000|      Chicago|
+-------------+---------------+-------+-------------+

root
 |-- department_id: integer (nullable = true)
 |-- department_name: string (nullable = true)
 |-- budget: integer (nullable = true)
 |-- location: string (nullable = true)



---
## The Spark UI

Open **http://localhost:4040** in your browser while this notebook is running. You'll see:
- **Jobs** — each action (`.show()`, `.count()`) created a job
- **Stages** — jobs are broken into stages at shuffle boundaries
- **SQL/DataFrame** — query plans for DataFrame operations
- **Environment** — Spark configuration

Every `.show()` and `.count()` you ran above created entries. Go look!

---
## Capstone Exercise

Combine everything you learned in this module:

1. Read all 3 CSV files (`employees.csv`, `departments.csv`, `sales.csv`)
2. Print the row count for each
3. Create a new DataFrame from Python data with columns `city`, `country`, `population` (at least 4 rows)
4. Use `.describe()` on the employees DataFrame (it shows summary statistics — count, mean, stddev, min, max)
5. Stop the SparkSession when done

In [ ]:
# Your capstone code here


In [17]:
# --- Capstone Solution ---
employees = spark.read.csv("../data/employees.csv", header=True, inferSchema=True)
departments = spark.read.csv("../data/departments.csv", header=True, inferSchema=True)
sales = spark.read.csv("../data/sales.csv", header=True, inferSchema=True)

print(f"Employees: {employees.count()} rows")
print(f"Departments: {departments.count()} rows")
print(f"Sales: {sales.count()} rows")

cities = [
    ("Tokyo", "Japan", 13960000),
    ("London", "UK", 8982000),
    ("New York", "USA", 8336000),
    ("Paris", "France", 2161000),
]
cities_df = spark.createDataFrame(cities, ["city", "country", "population"])
cities_df.show()

print("Employee summary statistics:")
employees.describe().show()

spark.stop()

Employees: 30 rows
Departments: 6 rows
Sales: 100 rows
+--------+-------+----------+
|    city|country|population|
+--------+-------+----------+
|   Tokyo|  Japan|  13960000|
|  London|     UK|   8982000|
|New York|    USA|   8336000|
|   Paris| France|   2161000|
+--------+-------+----------+

Employee summary statistics:
+-------+-----------------+-------------+------------------+------------------+-------------+
|summary|      employee_id|         name|     department_id|            salary|         city|
+-------+-----------------+-------------+------------------+------------------+-------------+
|  count|               30|           30|                30|                30|           30|
|   mean|             15.5|         NULL|103.16666666666667| 81833.33333333333|         NULL|
| stddev|8.803408430829505|         NULL|1.7436255002314767|15168.213891018222|         NULL|
|    min|                1| Aaron Brooks|               101|             61000|      Chicago|
|    max|        

---
## What You Learned

- **SparkSession** is the entry point — always create one first
- **SparkContext** gives low-level cluster info (parallelism, master URL)
- **DataFrames** are distributed tables — Spark's core abstraction
- **`.show()`** and **`.count()`** are **actions** that trigger execution
- **`.printSchema()`** shows column types without triggering a full computation
- **Spark UI** at port 4040 shows job execution details

Next: **Module 2 — RDDs**, where you'll understand what's under the hood of DataFrames.